In [1]:
using Pkg
Pkg.activate(".")
using Distributed
using CSV, DataFrames, BSON, Random

  Activating project at `c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned`


In [2]:
num_workers = 4            # ← set to number of CPU cores you want to use
num_replicates = 4        # ← set as desired
addprocs(num_workers)

4-element Vector{Int64}:
 2
 3
 4
 5

In [ ]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "data/median_RV.csv"
    data_column                  = "x1"
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 1000
    forecast_horizon             = 1
    forecast_length              = 2130
    random_seed                  = 1234

    smoothing_bandwidth          = 0.05
    cutoff_start_index           = 100

    benchmark_method             = "HAR"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.4
    kernel_type                  = "Epanechnikov"
    max_ar_order                 = 1
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.1
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.5
    smoothing_kernel             = "one-sided"
    kernel_type_tvEWD            = "Epanechnikov"
    kernel_type_tvHAR            = "Epanechnikov"
    kernel_type_tvAR             = "Epanechnikov"

    alpha_level                  = 0.05
    verbose_output               = true
end

UndefVarError: UndefVarError: `SEDThresholds` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [125]:
# load
df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame;

# turn column name into a Symbol, drop missings & scale
col_sym = Symbol(data_column);
series  = Float64.(df[.!ismissing.(df[!, col_sym]), col_sym]);

In [ ]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end;

# remove working processes
rmprocs(workers())

      From worker 27:	[ Info: Performing boostrap simulation number 6
      From worker 22:	[ Info: Performing boostrap simulation number 1
      From worker 25:	[ Info: Performing boostrap simulation number 4
      From worker 23:	[ Info: Performing boostrap simulation number 2
      From worker 24:	[ Info: Performing boostrap simulation number 3
      From worker 26:	[ Info: Performing boostrap simulation number 5
      From worker 22:	[ Info: Bootstrap 1 generated.
      From worker 25:	[ Info: Bootstrap 4 generated.
      From worker 22:	[ Info: Performing boostrap simulation number 7
      From worker 25:	[ Info: Performing boostrap simulation number 8
      From worker 26:	[ Info: Bootstrap 5 generated.
      From worker 26:	[ Info: Performing boostrap simulation number 9
      From worker 24:	[ Info: Bootstrap 3 generated.
      From worker 24:	[ Info: Performing boostrap simulation number 10
      From worker 27:	[ Info: Bootstrap 6 generated.
      From worker 27:	[ Info: Perf

Worker 24 terminated.
Unhandled Task ERROR: IOError: read: connection reset by peer (ECONNRESET)
Stacktrace:
  [1] wait_readnb(x::Sockets.TCPSocket, nb::Int64)
    @ Base .\stream.jl:410
  [2] (::Base.var"#wait_locked#832")(s::Sockets.TCPSocket, buf::IOBuffer, nb::Int64)
    @ Base .\stream.jl:981
  [3] unsafe_read(s::Sockets.TCPSocket, p::Ptr{UInt8}, nb::UInt64)
    @ Base .\stream.jl:987
  [4] unsafe_read
    @ .\io.jl:890 [inlined]
  [5] unsafe_read(s::Sockets.TCPSocket, p::Base.RefValue{NTuple{4, Int64}}, n::Int64)
    @ Base .\io.jl:889
  [6] read!
    @ .\io.jl:894 [inlined]
  [7] deserialize_hdr_raw
    @ C:\Users\mikul\AppData\Local\Programs\Julia-1.11.4\share\julia\stdlib\v1.11\Distributed\src\messages.jl:167 [inlined]
  [8] message_handler_loop(r_stream::Sockets.TCPSocket, w_stream::Sockets.TCPSocket, incoming::Bool)
    @ Distributed C:\Users\mikul\AppData\Local\Programs\Julia-1.11.4\share\julia\stdlib\v1.11\Distributed\src\process_messages.jl:172
  [9] process_tcp_streams(r

In [112]:
sed_vals

2-element Vector{Vector{Float64}}:
 [NaN, -1.2286464147200055e-5, 3.1688014433987384e-5, 5.326731074203238e-5, 7.786993335949329e-5, 6.544362428055366e-5, 4.432486011421513e-5, 3.377397209790851e-5, 5.6328507405204496e-5, 8.808226607472015e-6  …  1.0267379542004259e-5, 1.257293538733008e-5, 9.77516028516236e-6, 1.2191985233781615e-5, 9.883579758519479e-6, 3.4170002490606035e-6, -1.4398124947977305e-6, -2.163596275779172e-6, 2.6715849388465884e-7, 8.289293193757262e-6]
 [NaN, -4.648275094491989e-5, 4.814776149301456e-5, 7.215380773527744e-5, 3.64612060809129e-5, -1.57423536582127e-5, -3.970062948598554e-5, -2.8041458550088862e-5, -4.902593023191748e-5, -3.898535914650394e-6  …  -9.808409878288938e-6, -2.157332733758135e-6, 2.5662940416351046e-6, 3.7744624681437583e-6, 8.175932032387055e-6, -5.432293501262097e-7, 3.051948358683015e-6, 4.77642298687042e-6, 5.910122655581766e-6, 7.716650046639352e-6]

In [113]:
# 1) Count NaNs in each inner vector
nan_counts_per_series = map(v -> count(isnan, v), sed_vals)

# 2) Total number of NaNs across all series
total_nans = sum(nan_counts_per_series)

2

In [117]:
thr = SEDThresholds.compute_global_threshold(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

SED threshold: 7.853553124877703e-6


In [11]:
# will create sed_thresholds.bson in your working directory
BSON.@save "sed_thresholds.bson" sed_vals thr